In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")
import random
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt
from matplotlib_venn import venn2

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("takaito/atmacup17")

print("Path to dataset files:", path)

# 変数の設定

In [ ]:
class CFG:
    VER = 1
    AUTHOR = "ishizuka"
    COMPETITION = "atmacup17"
    DATA_PATH = Path("/workspace/kaggle_llm_book/ch3/data/takaito/atmacup17")
    SEED = 42
    N_SPLIT = 3
    TARGET_COL = "Recommended IND"
    TARGET_COL_CLASS_NUM = 2

# 乱数の設定

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
seed_everything(CFG.SEED)

# データの読み込み

In [ ]:
clothing_master_df = pd.read_csv(CFG.DATA_PATH / "clothing_master.csv")
train_df = pd.read_csv(CFG.DATA_PATH / "train.csv")
test_df = pd.read_csv(CFG.DATA_PATH / "test.csv")

# 読み込んだデータの確認

In [ ]:
clothing_master_df.head()

In [ ]:
train_df.head()

In [ ]:
test_df.head()

In [ ]:
train_df = train_df.merge(clothing_master_df, on="Clothing ID", how="left")
test_df = test_df.merge(clothing_master_df, on="Clothing ID", how="left")

# EDA

## カラム名

In [ ]:
train_cols = train_df.columns
test_cols = test_df.columns
for col in train_cols:
    if col in test_cols:
        print("train&test:", col)
    else:
        print("train only:", col)

In [ ]:
numerical_features = ["Age", "Positive Feedback Count"]
categorical_features = ["Clothing ID", "Division Name", "Department Name", "Class Name", "Title", "Review Text"]

## 量的変数

In [ ]:
for feature in numerical_features:
    plt.title(feature)
    train_df[feature].plot.kde(label="train")
    test_df[feature].plot.kde(label="test")
    plt.legend()
    plt.show()
    plt.close("all")

# 質的変数

In [ ]:
for feature in categorical_features:
    plt.title(feature)
    venn2([set(train_df[feature]), set(test_df[feature])])
    plt.show()
    plt.close("all")

## 数値特徴量の基本統計量

In [ ]:
for feature in numerical_features:
    print(f"=== {feature} ===")
    df_stats = pd.concat(
        [train_df[feature].describe(), test_df[feature].describe()],
        axis=1,
        keys=["train", "test"]
    )
    display(df_stats)
    print()

## Rating の分布とターゲット率の関係

In [ ]:
df_rating_agg = (
    train_df.groupby("Rating")[CFG.TARGET_COL]
    .agg(count="count", target_rate="mean")
    .reset_index()
)
df_rating_agg["target_rate(%)"] = (df_rating_agg["target_rate"] * 100).round(2)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(df_rating_agg["Rating"], df_rating_agg["count"])
axes[0].set_title("Rating - Count")
axes[0].set_xlabel("Rating")
axes[0].set_ylabel("Count")

axes[1].bar(df_rating_agg["Rating"], df_rating_agg["target_rate(%)"])
axes[1].set_title("Rating - Target Rate (%)")
axes[1].set_xlabel("Rating")
axes[1].set_ylabel("Target Rate (%)")

plt.tight_layout()
plt.show()
display(df_rating_agg.drop(columns="target_rate"))